In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Circuit Analysis Code Evaluation

This notebook evaluates the code implementation for circuit analysis in the repository located at `/net/scratch2/smallyan/InterpDetect_eval`.

## Setup and Configuration

In [2]:
# Load environment variables from .bashrc
import subprocess
result = subprocess.run(['bash', '-c', 'source /home/smallyan/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '=' in line:
        key, _, value = line.partition('=')
        os.environ[key] = value

# Set model cache directory
os.environ['HF_HOME'] = '/net/projects2/chai-lab/shared_models'
os.environ['TRANSFORMERS_CACHE'] = '/net/projects2/chai-lab/shared_models'

print("Environment configured")
print(f"HF_HOME: {os.environ.get('HF_HOME')}")

Environment configured
HF_HOME: /net/projects2/chai-lab/shared_models


In [3]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

CUDA available: True
CUDA device: NVIDIA A100 80GB PCIe
Number of GPUs: 1


## Code Evaluation

Based on the CodeWalkthrough.md file, the project implements a hallucination detection framework for RAG systems using interpretability techniques. The main components are:

### Core Analysis Scripts (Part 2: Training & Prediction)
1. **compute_scores.py** - Computes ECS (External Context Score) and PKS (Parametric Knowledge Score) using TransformerLens
2. **classifier.py** - Trains classifiers (LR, SVC, RandomForest, XGBoost) on the computed scores
3. **predict.py** - Runs predictions and evaluates at span and response levels

### Preprocessing Scripts (Part 1)
1. **preprocess.py** - Loads RAGBench data and adds prompt spans
2. **generate_response_gpt.py** - Generates responses using GPT models
3. **generate_labels.py** - Generates hallucination labels using LettuceDetect and LLM-as-a-judge
4. **filter.py** - Filters datasets based on LLM judge agreement
5. **helper.py** - Utility functions for text cleaning and semantic chunking

### Baseline Scripts (Part 3)
1. **run_gpt.py** - GPT baseline evaluation
2. **run_groq.py** - Groq (Llama) baseline evaluation
3. **run_hf.py** - HuggingFace models baseline evaluation
4. **run_ragas.py** - RAGAS baseline evaluation
5. **run_refchecker.py** - RefChecker baseline evaluation
6. **run_trulens.py** - TruLens baseline evaluation

---

## Part 1: Core Analysis Code Evaluation

I will evaluate the core analysis scripts by executing each function/block.

In [4]:
# Install required packages
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformer_lens', 'sentence_transformers', 'feature_engine', 'xgboost'], 
               capture_output=True)
print("Packages installed")

Packages installed


### 1. compute_scores.py - Core Functions Evaluation

In [5]:
# Block 1: Import statements from compute_scores.py
import torch
from transformers import AutoTokenizer
from transformer_lens import HookedTransformer
import json
from torch.nn import functional as F
from typing import Dict, List, Tuple
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import argparse
import sys
import os
import gc
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pointbiserialr

print("Block 1 (compute_scores.py imports): SUCCESS")
block1_runnable = "Y"
block1_correct = "Y"
block1_redundant = "N"
block1_irrelevant = "N"
block1_note = ""

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Block 1 (compute_scores.py imports): SUCCESS


In [6]:
# Block 2: load_examples function from compute_scores.py
def load_examples(file_path):
    """Load examples from JSONL file"""
    print(f"Loading examples from {file_path}...")
    
    try:
        examples = []
        with open(file_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                examples.append(data)
        
        print(f"Loaded {len(examples)} examples")
        return examples
    except Exception as e:
        print(f"Error loading examples: {e}")
        return []

# Test with existing data
test_path = "/net/scratch2/smallyan/InterpDetect_eval/scripts/preprocess/datasets/test/test1176_w_labels_filtered.jsonl"
if os.path.exists(test_path):
    examples = load_examples(test_path)
    print(f"First example keys: {examples[0].keys() if examples else 'No examples'}")
    block2_runnable = "Y"
    block2_correct = "Y"
    block2_note = ""
else:
    # Try another path
    test_path2 = "/net/scratch2/smallyan/InterpDetect_eval/scripts/preprocess/datasets/test/test.jsonl"
    if os.path.exists(test_path2):
        examples = load_examples(test_path2)
        print(f"First example keys: {examples[0].keys() if examples else 'No examples'}")
        block2_runnable = "Y"
        block2_correct = "Y"
        block2_note = ""
    else:
        print("Test file not found")
        block2_runnable = "N"
        block2_correct = "N"
        block2_note = "Test data file not found"

block2_redundant = "N"
block2_irrelevant = "N"
print(f"Block 2 (load_examples): Runnable={block2_runnable}")

Loading examples from /net/scratch2/smallyan/InterpDetect_eval/scripts/preprocess/datasets/test/test1176_w_labels_filtered.jsonl...
Loaded 256 examples
First example keys: dict_keys(['id', 'question', 'documents', 'documents_sentences', 'prompt', 'prompt_spans', 'num_tokens', 'response', 'response_spans', 'labels', 'hallucinated_llama-4-maverick-17b-128e-instruct', 'hallucinated_gpt-oss-120b', 'labels_llama', 'labels_gpt'])
Block 2 (load_examples): Runnable=Y


In [7]:
# Block 3: setup_models function from compute_scores.py
# Modified to load to GPU properly

def setup_models(model_name, hf_model_name, device="cuda"):
    """Setup tokenizer, model, and sentence transformer"""
    print(f"Setting up models: {model_name}, {hf_model_name}")
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
        
        model = HookedTransformer.from_pretrained(
            model_name,
            device="cpu",
            torch_dtype=torch.float16
        )
        model.to(device)
        
        bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5").to(device)
        
        return tokenizer, model, bge_model
    except Exception as e:
        print(f"Error setting up models: {e}")
        return None, None, None

# Test model setup with Qwen3-0.6B
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

try:
    tokenizer, model, bge_model = setup_models("qwen3-0.6b", "Qwen/Qwen3-0.6B", device)
    if model is not None:
        print(f"Model loaded successfully on {next(model.parameters()).device}")
        print(f"Model config: n_layers={model.cfg.n_layers}, n_heads={model.cfg.n_heads}")
        block3_runnable = "Y"
        block3_correct = "Y"
        block3_note = ""
    else:
        block3_runnable = "N"
        block3_correct = "N"
        block3_note = "Model setup returned None"
except Exception as e:
    print(f"Error: {e}")
    block3_runnable = "N"
    block3_correct = "N"
    block3_note = str(e)

block3_redundant = "N"
block3_irrelevant = "N"
print(f"Block 3 (setup_models): Runnable={block3_runnable}")

Using device: cuda
Setting up models: qwen3-0.6b, Qwen/Qwen3-0.6B


tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [8]:
# Check if models are loaded
print(f"Model type: {type(model)}")
print(f"Tokenizer type: {type(tokenizer)}")
print(f"BGE Model type: {type(bge_model)}")
print(f"Model device: {next(model.parameters()).device if model is not None else 'N/A'}")
print(f"Model config: n_layers={model.cfg.n_layers}, n_heads={model.cfg.n_heads}, n_ctx={model.cfg.n_ctx}")

In [9]:
# Check model status
print("Checking model status...")
print(f"block3_runnable: {block3_runnable}")